# Продвинутые паттерны рекурсии в UnifyWeaver

Этот блокнот демонстрирует четыре основных паттерна рекурсии, которые UnifyWeaver способен обнаруживать и оптимизировать:

1. **Хвостовая рекурсия (Tail Recursion)** — итеративные циклы с аккумуляторами
2. **Линейная рекурсия (Linear Recursion)** — единственный рекурсивный вызов с мемоизацией
3. **Древовидная рекурсия (Tree Recursion)** — множественные рекурсивные вызовы по частям структуры
4. **Взаимная рекурсия (Mutual Recursion)** — предикаты, циклически вызывающие друг друга

## Цели обучения

- Изучить различные паттерны рекурсии
- Увидеть, как UnifyWeaver обнаруживает и оптимизирует каждый паттерн
- Сравнить характеристики производительности
- Понять, когда применять каждый паттерн

## Настройка

Инициализация окружения UnifyWeaver.

In [ ]:
% Загрузить инициализацию
['../init'].

% Загрузить необходимые модули
use_module(unifyweaver(core/recursive_compiler)).
use_module(unifyweaver(core/advanced/pattern_matchers)).

## Паттерн 1: Хвостовая рекурсия (Tail Recursion)

Хвостовая рекурсия использует аккумулятор для передачи промежуточных результатов, а рекурсивный вызов является **последним действием** в функции.

### Пример: Подсчет элементов в списке

In [ ]:
% Определить хвосторекурсивный count_items
:- dynamic count_items/3.

% Базовый случай: пустой список, вернуть аккумулятор
count_items([], Acc, Acc).

% Рекурсивный случай: увеличить аккумулятор, рекурсивно обработать хвост
count_items([_|T], Acc, N) :-
    Acc1 is Acc + 1,
    count_items(T, Acc1, N).  % ← Хвостовая позиция!

### Тестирование в Prolog

In [ ]:
% Тест: подсчет элементов в [a,b,c,d,e]
\+ \+ (
    count_items([a,b,c,d,e], 0, _N),
    format('Count: ~w~n', [_N])
).

### Проверка распознавания паттерна

In [ ]:
% Проверить, распознано ли как хвостовая рекурсия
\+ \+ (
    is_tail_recursive_accumulator(count_items/3, _AccInfo),
    format('Tail recursive: ~w~n', [_AccInfo])
).

### Компиляция в Bash

In [ ]:
% Скомпилировать и сохранить
\+ \+ (
    compile_recursive(count_items/3, [], _BashCode),
    setup_call_cleanup(
        open('../output/count_items_demo.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('✓ Compiled count_items to Bash with tail recursion optimization')
).

### Тестирование сгенерированного кода Bash

In [ ]:
%%bash
source ../output/count_items_demo.sh
echo "Подсчет элементов в [a,b,c,d,e]:"
count_items "[a,b,c,d,e]" 0 ""

## Паттерн 2: Линейная рекурсия (Linear Recursion)

Линейная рекурсия содержит **ровно один** рекурсивный вызов в каждом правиле (clause), а вычисления выполняются после возврата из рекурсивного вызова.

### Пример: Факториал (Factorial)

In [ ]:
% Определить факториал
:- dynamic factorial/2.

% Базовый случай
factorial(0, 1).

% Рекурсивный случай: ровно ОДИН рекурсивный вызов
factorial(N, F) :-
    N > 0,
    N1 is N - 1,
    factorial(N1, F1),  % ← Один рекурсивный вызов
    F is N * F1.        % ← Вычисление после вызова

### Тестирование в Prolog

In [ ]:
% Тест: факториал 5
\+ \+ (
    factorial(5, _F),
    format('5! = ~w~n', [_F])
).

### Проверка распознавания паттерна

In [ ]:
% Проверить, распознано ли как линейная рекурсия
is_linear_recursive_streamable(factorial/2),
writeln('✓ Detected as linear recursion').

### Компиляция в Bash

In [ ]:
% Скомпилировать и сохранить
\+ \+ (
    compile_recursive(factorial/2, [], _BashCode),
    % Сохранять только определения функций; Brush обрабатывает подключенные через source скрипты как прямое выполнение
    split_string(_BashCode, "\n", "\r", _BashLines),
    append(_LibraryLines, ["# Auto-execute when run directly (not when sourced)"|_], _BashLines),
    atomics_to_string(_LibraryLines, "\n", _LibraryCode),
    setup_call_cleanup(
        open('../output/factorial_demo.sh', write, _Stream),
        write(_Stream, _LibraryCode),
        close(_Stream)),
    writeln('✓ Compiled factorial to Bash with fold-based linear recursion')
).

### Тестирование сгенерированного кода Bash

In [ ]:
%%bash
source ../output/factorial_demo.sh
echo "Факториал 5:"
factorial 5 ""
echo ""
echo "Факториал 10:"
factorial 10 ""

## Паттерн 3: Древовидная рекурсия (Tree Recursion)

Древовидная рекурсия выполняет **несколько** рекурсивных вызовов для обработки различных частей структуры.

### Пример: Сумма элементов дерева (Tree Sum)

In [ ]:
% Определить tree_sum для бинарных деревьев
% Формат дерева: [Значение, ЛевоеПоддерево, ПравоеПоддерево] или []
:- dynamic tree_sum/2.

% Базовый случай: пустое дерево имеет сумму 0
tree_sum([], 0).

% Рекурсивный случай: сумма = значение + сумма_слева + сумма_справа
tree_sum([V, L, R], Sum) :-
    tree_sum(L, LS),   % ← Первый рекурсивный вызов
    tree_sum(R, RS),   % ← Второй рекурсивный вызов
    Sum is V + LS + RS.

### Тестирование в Prolog

In [ ]:
% Тест: tree_sum для [5, [3, [1, [], []], []], [2, [], []]]
%       5
%      / \
%     3   2
%    /
%   1
\+ \+ (
    tree_sum([5, [3, [1, [], []], []], [2, [], []]], _Sum),
    format('Tree sum: ~w (expected 11)~n', [_Sum])
).

### Компиляция в Bash

In [ ]:
% Скомпилировать и сохранить
\+ \+ (
    compile_recursive(tree_sum/2, [], _BashCode),
    setup_call_cleanup(
        open('../output/tree_sum_demo.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('✓ Compiled tree_sum to Bash with tree recursion')
).

### Тестирование сгенерированного кода Bash

In [ ]:
%%bash
source ../output/tree_sum_demo.sh
echo "Сумма элементов дерева [5,[3,[1,[],[]],[]],[2,[],[]]]:"
tree_sum "[5,[3,[1,[],[]],[]],[2,[],[]]]"

## Паттерн 4: Взаимная рекурсия (Mutual Recursion)

Взаимная рекурсия возникает, когда два или более предиката вызывают друг друга по циклу.

### Пример: Четное (Even) и Нечетное (Odd)

In [ ]:
% Определить взаимно рекурсивные is_even и is_odd
:- dynamic is_even/1.
:- dynamic is_odd/1.

% Базовый случай is_even
is_even(0).

% Рекурсивный is_even: N четно, если N-1 нечетно
is_even(N) :-
    N > 0,
    N1 is N - 1,
    is_odd(N1).  % ← Вызывает is_odd

% Базовый случай is_odd
is_odd(1).

% Рекурсивный is_odd: N нечетно, если N-1 четно
is_odd(N) :-
    N > 1,
    N1 is N - 1,
    is_even(N1).  % ← Вызывает is_even

### Тестирование в Prolog

In [ ]:
% Тестирование чет/нечет
is_even(0), writeln('✓ 0 is even').
is_even(4), writeln('✓ 4 is even').
is_odd(3), writeln('✓ 3 is odd').
is_odd(7), writeln('✓ 7 is odd').

### Проверка взаимной рекурсии

In [ ]:
% Построить граф вызовов и найти SCC
\+ \+ (
    use_module(unifyweaver(core/advanced/call_graph)),
    use_module(unifyweaver(core/advanced/scc_detection)),

    build_call_graph([is_even/1, is_odd/1], _Graph),
    format('Call graph: ~w~n', [_Graph]),

    find_sccs(_Graph, _SCCs),
    format('SCCs (mutual recursion groups): ~w~n', [_SCCs])
).

### Компиляция в Bash

In [ ]:
% Скомпилировать группу взаимной рекурсии
\+ \+ (
    use_module(unifyweaver(core/advanced/mutual_recursion)),

    compile_mutual_recursion([is_even/1, is_odd/1], [], _BashCode),
    split_string(_BashCode, "\n", "\r", _BashLines),
    append(_LibraryLines, ["# Main dispatch: route command line calls to functions"|_], _BashLines),
    atomics_to_string(_LibraryLines, "\n", _LibraryCode),
    setup_call_cleanup(
        open('../output/even_odd_demo.sh', write, _Stream),
        write(_Stream, _LibraryCode),
        close(_Stream)),
    writeln('✓ Compiled is_even/is_odd to Bash with shared memoization')
).

### Тестирование сгенерированного кода Bash

In [ ]:
%%bash
source ../output/even_odd_demo.sh
echo "Тестирование is_even и is_odd:"
is_even 0 >/dev/null && echo "✓ 0 четно"
is_even 4 >/dev/null && echo "✓ 4 четно"
is_odd 3 >/dev/null && echo "✓ 3 нечетно"
is_odd 7 >/dev/null && echo "✓ 7 нечетно"
is_even 5 >/dev/null 2>&1 || echo "✓ 5 не является четным"

## Сравнение паттернов

Сравним ключевые характеристики каждого паттерна:

| Паттерн | Рекурсивные вызовы | Оптимизация | Пространственная сложность | Область применения |
|:--------|:----------------|:-------------|:-----------------|:---------|
| **Хвостовая** | 1 (в хвостовой позиции) | Итеративный цикл | O(1) | Аккумуляторы, линейные обходы |
| **Линейная** | 1 (в любой позиции) | Свертка (fold) + мемоизация | O(n) таблица мемоизации | Числа Фибоначчи, факториал |
| **Древовидная** | 2+ (по структуре) | Структурная декомпозиция | O(глубина) стек | Операции над деревьями и графами |
| **Взаимная** | 1+ (между предикатами) | Общая мемоизация | O(n) общая таблица | Чет/нечет, взаимные определения |

## Порядок распознавания паттернов

UnifyWeaver пытается сопоставить паттерны в следующем порядке:

1. **Хвостовая рекурсия** (наиболее эффективная)
2. **Линейная рекурсия** (если не запрещена)
3. **Древовидная рекурсия** (структурная)
4. **Взаимная рекурсия** (поиск SCC)
5. **Базовая рекурсия** (стандартный резервный вариант)

Вы можете управлять сопоставлением с помощью `forbid_linear_recursion/1`.

## Упражнение: Ваша очередь!

Попробуйте определить и скомпилировать следующие предикаты:

### 1. Хвосторекурсивная сумма списка
```prolog
sum_list([], Acc, Acc).
sum_list([H|T], Acc, Sum) :-
    Acc1 is Acc + H,
    sum_list(T, Acc1, Sum).
```

### 2. Линейно-рекурсивные числа Фибоначчи
```prolog
fib(0, 0).
fib(1, 1).
fib(N, F) :-
    N > 1,
    N1 is N - 1,
    N2 is N - 2,
    fib(N1, F1),
    fib(N2, F2),
    F is F1 + F2.
```

### 3. Высота дерева
```prolog
tree_height([], 0).
tree_height([_, L, R], H) :-
    tree_height(L, HL),
    tree_height(R, HR),
    H is max(HL, HR) + 1.
```

In [ ]:
% Ваш код здесь!


## Резюме

В этом блокноте вы изучили:

✅ Четыре основных паттерна рекурсии в UnifyWeaver

✅ Как определять каждый паттерн на Prolog

✅ Как UnifyWeaver обнаруживает и оптимизирует каждый паттерн

✅ Характеристики производительности каждого паттерна

✅ Когда применять каждый из паттернов

## Следующие шаги

Переходите к **Блокноту 3: Визуализация графа вызовов**, чтобы изучить методы углубленного анализа и визуализации кода!